In [ ]:
OPENAI_API_KEY=""

In [148]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

### Loading

In [149]:
pip install docling

Note: you may need to restart the kernel to use updated packages.


In [150]:
# from docling.document_converter import DocumentConverter

# source = "https://arxiv.org/pdf/2408.09869"
# converter = DocumentConverter()
# doc = converter.convert(source).document
# print(doc.export_to_markdown())

In [151]:
from langchain_community.document_loaders import PyPDFLoader

In [152]:
pdf_path = r"C:\Users\SSD\Downloads\COMM-Drug-Testing-Policy.pdf"
print(f"\n📄 Loading: {pdf_path}")


📄 Loading: C:\Users\SSD\Downloads\COMM-Drug-Testing-Policy.pdf


In [153]:
loader = PyPDFLoader(pdf_path)

In [154]:
loader

In [155]:
documents = loader.load()
    


In [156]:
documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-09-18T00:30:17+05:30', 'moddate': '2025-09-19T10:12:34-04:00', 'subject': 'This policy defines the daily and annual limits for presumptive drug testing codes (codes 80305, 80306, 80307, and H0003) and definitive drug testing codes (G0480, G0481, G0482, G0483, G0659, 0006U, 0007U, 0011U, and 0020U) and addresses Specimen Validity Testing. Flag: DTP', 'title': 'Drug Testing Policy, Professional - Reimbursement Policy - UnitedHealthcare Commercial Plans and Individual Exchange', 'source': 'C:\\Users\\SSD\\Downloads\\COMM-Drug-Testing-Policy.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='UnitedHealthcare® Commercial and Individual Exchange \nReimbursement Policy  \nCMS 1500 \nPolicy Number 2025R6005B \nProprietary information of UnitedHealthcare.  Copyright 2025 United HealthCare Services, Inc \n \n \nDrug Testing Policy, Professio

In [157]:
question="give me list of presumptive drug names"

In [158]:
print(f"✅ Loaded {len(documents)} pages")
print(f"\n📖 Page 1 Preview:")
print("-" * 50)
print(documents[0].page_content[:400] + "...")
print(f"\n📋 Metadata: {documents[0].metadata}")
    

✅ Loaded 4 pages

📖 Page 1 Preview:
--------------------------------------------------
UnitedHealthcare® Commercial and Individual Exchange 
Reimbursement Policy  
CMS 1500 
Policy Number 2025R6005B 
Proprietary information of UnitedHealthcare.  Copyright 2025 United HealthCare Services, Inc 
 
 
Drug Testing Policy, Professional 
 
 
IMPORTANT NOTE ABOUT THIS REIMBURSEMENT POLICY 
You are responsible for submission of accurate claims.  This reimbursement policy is intended to ensur...

📋 Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-09-18T00:30:17+05:30', 'moddate': '2025-09-19T10:12:34-04:00', 'subject': 'This policy defines the daily and annual limits for presumptive drug testing codes (codes 80305, 80306, 80307, and H0003) and definitive drug testing codes (G0480, G0481, G0482, G0483, G0659, 0006U, 0007U, 0011U, and 0020U) and addresses Specimen Validity Testing. Flag: DTP', 'title': 'Drug Testing P

# Chunking

In [159]:
from langchain_text_splitters import  RecursiveCharacterTextSplitter

In [160]:
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

In [161]:
print(f"\n⚙️  Settings:")
print(f"   chunk_size: 1000")
print(f"   chunk_overlap: 200")
    
chunks = text_splitter.split_documents(documents)


⚙️  Settings:
   chunk_size: 1000
   chunk_overlap: 200


In [162]:
print(f"\n✅ Created {len(chunks)} chunks from {len(documents)} pages")
    
# Statistics
lengths = [len(c.page_content) for c in chunks]
print(f"\n📊 Chunk Statistics:")
print(f"   Min: {min(lengths)} chars")
print(f"   Max: {max(lengths)} chars")
print(f"   Avg: {sum(lengths)//len(lengths)} chars")


✅ Created 17 chunks from 4 pages

📊 Chunk Statistics:
   Min: 154 chars
   Max: 997 chars
   Avg: 850 chars


In [163]:
from langchain_openai import OpenAIEmbeddings
import numpy as np
embeddings = OpenAIEmbeddings()

In [126]:
pip install -qU "langchain-chroma>=0.1.2"

Note: you may need to restart the kernel to use updated packages.


In [164]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [166]:
ids = vector_store.add_documents(documents=chunks)

In [128]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [167]:
vector_store.similarity_search(question,k=4)

[Document(id='f09ac288-15bd-44d9-84f9-6ec2512fa644', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-09-18T00:30:17+05:30', 'moddate': '2025-09-19T10:12:34-04:00', 'subject': 'This policy defines the daily and annual limits for presumptive drug testing codes (codes 80305, 80306, 80307, and H0003) and definitive drug testing codes (G0480, G0481, G0482, G0483, G0659, 0006U, 0007U, 0011U, and 0020U) and addresses Specimen Validity Testing. Flag: DTP', 'title': 'Drug Testing Policy, Professional - Reimbursement Policy - UnitedHealthcare Commercial Plans and Individual Exchange', 'source': 'C:\\Users\\SSD\\Downloads\\COMM-Drug-Testing-Policy.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='This policy enforces the code description for presumptive and definitive drug testing in that the service should be \nreported once per day and it includes Specimen Validity Testing.   \n \nClinical drug t

In [130]:
vector_store.similarity_search?

Signature:
vector_store.similarity_search(
    query: 'str',
    k: 'int' = 4,
    filter: 'dict[str, str] | None' = None,
    **kwargs: 'Any',
) -> 'list[Document]'
Docstring:
Run similarity search with Chroma.

Args:
    query: Query text to search for.
    k: Number of results to return.
    filter: Filter by metadata.
    kwargs: Additional keyword arguments to pass to Chroma collection query.

Returns:
    List of documents most similar to the query text.
File:      c:\users\ssd\anaconda3\lib\site-packages\langchain_chroma\vectorstores.py
Type:      method

In [131]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000020DEDE8FED0>, search_kwargs={'k': 4})

#### prompt

In [135]:
###prompt
stuff_template = """Answer based on the context below.
    
Context: {context}

Question: {question}

Answer:"""

In [136]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [137]:
stuff_prompt = ChatPromptTemplate.from_template(stuff_template)

In [138]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [139]:
def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

In [140]:
stuff_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | stuff_prompt
        | llm
    )

In [141]:
stuff_chain.invoke(question)

AIMessage(content='1. Acetaminophen\n2. Ibuprofen\n3. Omeprazole\n4. Simvastatin\n5. Metformin\n6. Lisinopril\n7. Albuterol\n8. Atorvastatin\n9. Levothyroxine\n10. Amlodipine', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 32, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWXjQaBLq1oCXhRJLAnlZseDHJHwM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da880-c31c-7082-aebc-381754638d64-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 71, 'total_tokens': 103, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reason